# Kapitel 17 Begleit-Notebook
**Build Your First LLM — Kapitel 17: Deployment-Optionen**

Dieses Notebook bündelt die ausführbaren Code-Beispiele aus Kapitel 17. Modal-Deployment läuft über Ihr Terminal, nicht über Colab, aber dieses Notebook erklärt jeden Schritt.

- Installationen: modal
- Daten: Inline-Beispiele; keine externen Dateien erforderlich
- Laufzeit: Die meisten Modal-Befehle laufen im Terminal, nicht im Notebook

In [ ]:
# ===== SETUP =====
# Modal installieren
!pip install -q modal

print('Modal installiert!')
print('WICHTIG: Führen Sie "modal setup" in Ihrem Terminal aus, um sich zu authentifizieren.')
print('Dies öffnet ein Browserfenster für den Login.')

## Abschnitt 17.1: Warum Modal?

**Was ist Serverless?**

| Traditioneller Server | Serverless (Modal) |
|-------------------|-------------------|
| Sie mieten einen Computer 24/7 | Code läuft nur bei Bedarf |
| Sie zahlen auch im Leerlauf | Sie zahlen pro Sekunde Rechenzeit |
| Sie verwalten Updates, Sicherheit | Die Plattform übernimmt alles |
| Feste Kapazität | Automatische Skalierung |

**Analogie:** Traditionelles Hosting ist wie ein eigenes Auto zu besitzen. Serverless ist wie ein Taxi zu nutzen – Sie zahlen nur, wenn Sie fahren.

**Warum Modal für LLMs?**
- GPU-Unterstützung eingebaut (T4 bis H100)
- 30$/Monat kostenlose Credits (~50 Stunden T4-GPU-Zeit)
- Python-nativ (kein Docker, kein YAML)
- Skaliert auf Null (keine Gebühren im Leerlauf)

## Abschnitt 17.2: Hallo Modal

Die einfachste Modal-App. Speichern Sie dies als `hello.py` und führen Sie es mit `modal run hello.py` aus:

In [ ]:
# Speichern Sie dies als hello.py
hello_modal_code = '''
import modal

app = modal.App("hello-world")

@app.function()
def hello(name: str) -> str:
    return f"Hallo, {name}!"

@app.local_entrypoint()
def main():
    result = hello.remote("Welt")
    print(result)
'''

print("Speichern Sie diesen Code als 'hello.py':")
print(hello_modal_code)
print("\nDann ausführen: modal run hello.py")

**Was ist gerade passiert?**

1. `modal.App()` erstellt Ihre Anwendung
2. `@app.function()` markiert Code, der in Modals Cloud ausgeführt werden soll
3. `hello.remote()` ruft die Funktion remote auf
4. Modal hat einen Container gestartet, Ihren Code ausgeführt und das Ergebnis zurückgegeben

## Abschnitt 17.3: GPU-Unterstützung hinzufügen

Ein Parameter fügt GPU-Unterstützung hinzu:

In [ ]:
# Speichern Sie dies als gpu_test.py
gpu_test_code = '''
import modal

app = modal.App("gpu-test")

@app.function(gpu="T4")  # Das ist alles!
def check_gpu():
    import torch
    if torch.cuda.is_available():
        device = torch.cuda.get_device_name(0)
        return f"GPU verfügbar: {device}"
    return "Keine GPU gefunden"

@app.local_entrypoint()
def main():
    print(check_gpu.remote())
'''

print("Speichern Sie diesen Code als 'gpu_test.py':")
print(gpu_test_code)
print("\nDann ausführen: modal run gpu_test.py")
print("\nErwartete Ausgabe: GPU verfügbar: Tesla T4")

**Ein Parameter.** Keine CUDA-Installation, keine Treiberverwaltung, keine Docker-Images. Modal übernimmt den gesamten GPU-Stack.

## Abschnitt 17.4: Abhängigkeiten installieren

Ihr LLM benötigt Bibliotheken. Modal ermöglicht es Ihnen, Ihre Container-Umgebung in Python zu definieren:

In [ ]:
# Speichern Sie dies als with_deps.py
with_deps_code = '''
import modal

# Container-Image definieren
image = modal.Image.debian_slim(python_version="3.11").pip_install(
    "torch",
    "transformers",
    "accelerate",
)

app = modal.App("with-deps")

@app.function(image=image, gpu="T4")
def generate_text():
    from transformers import pipeline
    
    generator = pipeline("text-generation", model="gpt2", device=0)
    result = generator("Der Sinn des Lebens ist", max_length=50)
    return result[0]["generated_text"]

@app.local_entrypoint()
def main():
    print(generate_text.remote())
'''

print("Speichern Sie diesen Code als 'with_deps.py':")
print(with_deps_code)
print("\nDann ausführen: modal run with_deps.py")

Der `image`-Parameter teilt Modal mit, was installiert werden soll. Der erste Durchlauf dauert länger (Image wird erstellt), aber nachfolgende Durchläufe verwenden das gecachte Image wieder.

## Abschnitt 17.5: Web-Endpunkte mit FastAPI

Erstellen Sie einen Web-Endpunkt, auf den jeder zugreifen kann. Hier zahlt sich das FastAPI-Wissen aus Kapitel 16 aus:

In [ ]:
# Speichern Sie dies als simple_api.py
simple_api_code = '''
import modal
from fastapi import FastAPI

app = modal.App("my-api")
web_app = FastAPI()

@web_app.get("/health")
def health():
    return {"status": "healthy"}

@web_app.get("/hello/{name}")
def hello(name: str):
    return {"message": f"Hallo, {name}!"}

@app.function()
@modal.asgi_app()
def serve():
    return web_app
'''

print("Speichern Sie diesen Code als 'simple_api.py':")
print(simple_api_code)
print("\nDann ausführen: modal deploy simple_api.py")
print("\nSie erhalten eine URL wie: https://your-workspace--my-api-serve.modal.run")

**Ihre API ist live!** Öffnen Sie diese URL in Ihrem Browser. Fügen Sie `/health` zum Pfad hinzu:

```
https://your-workspace--my-api-serve.modal.run/health
```

Sie sollten sehen: `{"status": "healthy"}`

## Abschnitt 17.6: Der vollständige LLM-Service

Hier ist ein produktionsreifes LLM-Deployment:

In [ ]:
# Speichern Sie dies als llm_service.py
llm_service_code = '''
import modal
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import os

# Container-Image mit LLM-Abhängigkeiten definieren
image = modal.Image.debian_slim(python_version="3.11").pip_install(
    "fastapi",
    "vllm==0.6.4",  # Version fixieren für Reproduzierbarkeit
    "torch",
)

app = modal.App("my-llm-service")
web_app = FastAPI(title="My LLM API", version="1.0.0")

# Request/Response-Modelle (bekannt aus Kapitel 16)
class ChatRequest(BaseModel):
    message: str
    max_tokens: int = 256

class ChatResponse(BaseModel):
    response: str

# Globale Modellreferenz (einmal pro Container geladen)
_model = None

def get_model():
    """Modell einmal laden, für alle Anfragen wiederverwenden."""
    global _model
    if _model is None:
        from vllm import LLM
        _model = LLM(
            model="Qwen/Qwen2.5-1.5B-Instruct",
            trust_remote_code=True,
        )
    return _model

@web_app.get("/health")
def health():
    """Health-Check-Endpunkt (Kapitel 16 Rückblick)."""
    return {"status": "healthy", "model": "Qwen2.5-1.5B-Instruct"}

@web_app.post("/chat", response_model=ChatResponse)
def chat(request: ChatRequest):
    """Eine Antwort auf eine Nachricht generieren."""
    try:
        model = get_model()
        from vllm import SamplingParams
        
        params = SamplingParams(max_tokens=request.max_tokens)
        outputs = model.generate([request.message], params)
        response_text = outputs[0].outputs[0].text
        
        return ChatResponse(response=response_text)
    except Exception as e:
        raise HTTPException(500, f"Generierung fehlgeschlagen: {str(e)}")

@app.function(
    image=image,
    gpu="T4",
    timeout=300,
    scaledown_window=300,  # 5 Minuten warm halten
)
@modal.concurrent(max_inputs=10)  # Mehrere Anfragen pro Container verarbeiten
@modal.asgi_app()
def serve():
    return web_app
'''

print("Speichern Sie diesen Code als 'llm_service.py':")
print(llm_service_code)

In [ ]:
print("Deployment mit: modal deploy llm_service.py")
print("\nErstes Deployment dauert einige Minuten (Modell wird heruntergeladen).")
print("Nachfolgende Deployments sind schnell.")
print("\nTesten mit:")
print('curl https://your-url.modal.run/health')
print('curl -X POST https://your-url.modal.run/chat -H "Content-Type: application/json" -d \'{"message": "Was ist Python?"}\')')

## Abschnitt 17.7: Secrets und Volumes

### Secrets erstellen

```bash
# Ein Secret über die Kommandozeile erstellen
modal secret create my-secrets API_KEY=your_secret_key
```

### Secrets im Code verwenden

In [ ]:
# Secrets in Ihrem Code verwenden
secrets_example = '''
@app.function(secrets=[modal.Secret.from_name("my-secrets")])
def with_secrets():
    import os
    api_key = os.environ["API_KEY"]
    # Den Schlüssel sicher verwenden...
'''

print("Secrets verwenden:")
print(secrets_example)

In [ ]:
# Volumes zum Cachen von Modellen verwenden
volumes_example = '''
# Ein Volume für Modell-Cache erstellen
model_cache = modal.Volume.from_name("model-cache", create_if_missing=True)

@app.function(
    gpu="T4",
    volumes={"/root/.cache/huggingface": model_cache},
)
def with_cache():
    # Modelle werden einmal heruntergeladen, dann im Volume gecacht
    pass
'''

print("Volumes zum Cachen von Modellen verwenden:")
print(volumes_example)
print("\nErste Anfrage lädt das Modell herunter. Nachfolgende Anfragen verwenden den Cache.")

## Abschnitt 17.8: GPU-Optionen

| Modellgröße | Empfohlene GPU | Kosten/Stunde |
|------------|-----------------|----------|
| < 3B Parameter | T4 (16GB) | 0,59$ |
| 3-8B Parameter | A10G (24GB) | 1,10$ |
| 8-30B Parameter | A100-40GB | 2,50$ |
| 30B+ Parameter | A100-80GB / H100 | 4-8$ |

**Beginnen Sie mit T4.** Upgraden Sie nur bei Bedarf.

In [ ]:
# GPU-Auswahl-Beispiele
print("GPU-Auswahl ist ein Parameter:")
print()
print('@app.function(gpu="T4")      # Budget-Option')
print('@app.function(gpu="A10G")    # Mittelklasse')
print('@app.function(gpu="A100")    # Hohe Leistung')
print('@app.function(gpu="H100")    # Maximale Leistung')
print('@app.function(gpu="A100:2")  # Zwei A100s')

## Abschnitt 17.9: Ihr Deployment verwalten

### Logs anzeigen

```bash
modal app logs my-llm-service
```

Oder verwenden Sie das Dashboard unter [modal.com](https://modal.com).

### Ihre App aktualisieren

```bash
# Einfach neu deployen
modal deploy llm_service.py
```

Keine Ausfallzeit. Modal übernimmt automatisch Rolling Updates.

### Kostenkontrolle

- Verwenden Sie `scaledown_window`, um inaktive Container herunterzufahren
- Beginnen Sie mit T4, upgraden Sie nur bei Bedarf
- Überwachen Sie Ausgaben im Modal-Dashboard

## Zusammenfassung

Sie haben gelernt, wie man:

1. **Funktionen zu Modal deployed** mit `@app.function()`
2. **GPU-Unterstützung hinzufügt** mit einem einzigen Parameter: `gpu="T4"`
3. **Abhängigkeiten installiert** mit `modal.Image`
4. **Web-Endpunkte erstellt** mit FastAPI + `@modal.asgi_app()`
5. **Secrets sicher speichert** mit `modal.Secret`
6. **Modelle cacht** mit `modal.Volume`
7. **Kosten kontrolliert** mit Idle-Timeouts und Monitoring

**Ihr LLM ist nicht mehr auf Ihrem Laptop gefangen.** Es ist für jeden zugänglich, überall, mit GPU-Beschleunigung.

**Befehle zum Merken:**
```bash
pip install modal      # Installieren
modal setup            # Authentifizieren
modal run file.py      # Einmal ausführen
modal deploy file.py   # Permanent deployen
modal app logs name    # Logs anzeigen
```